# Self-Attention from Scratch: Manual Q, K, V

This notebook builds **single-head scaled dot-product self-attention**
by hand (Vaswani et al., 2017, "Attention Is All You Need", Eq. 1):

$$\text{Attention}(Q,K,V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

For self-attention, $Q,K,V$ are linear projections of the same input
token sequence $X$ (no bias, matching the paper):

$$Q = XW^Q,\quad K = XW^K,\quad V = XW^V$$

| symbol | code variable | shape | meaning |
|---|---|---|---|
| $X$ | `x` | `[B,N,D]` | input token embeddings (B=batch, N=#tokens, D=embed dim) |
| $W^Q,W^K$ | `W_q.weight`, `W_k.weight` | `[d_k,D]` | learned query/key projections |
| $W^V$ | `W_v.weight` | `[d_v,D]` | learned value projection |
| $Q,K$ | `q,k` | `[B,N,d_k]` | projected queries/keys |
| $V$ | `v` | `[B,N,d_v]` | projected values |
| $QK^\top$ | `raw_scores` | `[B,N,N]` | raw similarity between every query/key pair |
| $\sqrt{d_k}$ | `scale` | scalar | stabilizes softmax gradient (paper Sec. 3.2.1) |
| $\text{softmax}(\cdot)$ | `attention_weights` | `[B,N,N]` | row $i$ sums to 1: how much query $i$ attends to each key $j$ |
| output | `output` | `[B,N,d_v]` | weighted sum of value vectors per query |

**What "manual" means here**: $Q,K,V$ come from `nn.Linear` projections
(standard), but the scaled dot product, the softmax (explicit
`exp`/`sum` with max-subtraction, not `F.softmax`), and the weighted
sum over $V$ are all written out as plain tensor ops -- no
`nn.MultiheadAttention`, no `F.scaled_dot_product_attention`.

**Interpretation caveat**: $W^Q,W^K,W^V$ below are randomly initialized
and never trained. The toy "sentence" has category structure (animals /
vehicles / fruits) purely so $X$ is not pure noise -- the attention
heatmap demonstrates the *mechanism* of Eq. 1 (shapes, softmax
normalization, weighted combination of $V$), not learned semantic
attention.

Core code lives in `code/self_attention_qkv.py`
(`SelfAttentionQKV`, `manual_softmax`, `make_toy_token_sequence`,
`print_all_shapes`, `worked_example_reconstruct_output`,
`visualize_attention_matrix`), unit-checked with
`python code/self_attention_qkv.py --smoke-test`. This notebook is a
thin orchestration layer.

## 1. Clone/pull repository

Self-attention code lives in `code/self_attention_qkv.py`, not duplicated in this notebook. Replace `REPO_URL` with this repository's URL.

In [ ]:
import os

REPO_URL = "https://github.com/RoeiZ/from_zero_to_hero.git"
REPO_DIR = "/content/from_zero_to_hero"

if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull

import sys
sys.path.insert(0, os.path.join(REPO_DIR, "code"))


## 2. Build a toy token sequence (no dataset)

Six tokens from three categories (`cat`/`dog`, `car`/`truck`, `apple`/`banana`), embedded as a category base vector plus small noise -- gives $X$ structure without downloading any dataset.

In [ ]:
import torch

from self_attention_qkv import (
    SelfAttentionQKV,
    make_toy_token_sequence,
    print_all_shapes,
    worked_example_reconstruct_output,
    visualize_attention_matrix,
)

x, tokens = make_toy_token_sequence(seed=0)  # x: [1,N,8]
print(f"x.shape = {tuple(x.shape)}")
print(f"tokens = {tokens}")


## 3. Run self-attention and print every intermediate tensor shape

`SelfAttentionQKV` projects `x` to `q,k,v`, computes `raw_scores = Q K^T`, scales by `1/sqrt(d_k)`, applies the manual softmax, and returns every named intermediate.

In [ ]:
torch.manual_seed(0)
HEAD_DIM = 4

module = SelfAttentionQKV(embed_dim=x.shape[-1], head_dim=HEAD_DIM)
intermediates = module(x)

print_all_shapes(x, intermediates)


## 4. Worked example: reconstruct one output token by hand

For one query token, print its attention weight onto every key token (sorted, most-attended first), then manually recompute
$$\text{output}[i] = \sum_j \text{attention\_weights}[i,j]\, v[j]$$
as an explicit weighted sum over value rows, and check it matches the module's own output row exactly.

In [ ]:
attention_weights = intermediates["attention_weights"][0]  # [N,N], single batch element

manual_output_row = worked_example_reconstruct_output(
    intermediates["v"][0],
    attention_weights,
    intermediates["output"][0],
    tokens,
    query_index=0,
)


## 5. Visualize the attention matrix

Heatmap of `attention_weights[i,j]`: rows are query tokens, columns are key tokens, each row sums to 1. See the interpretation caveat above -- the pattern reflects untrained random projections, not learned semantics.

In [ ]:
visualize_attention_matrix(attention_weights, tokens)
